# Pipeline tests: idempotency, upserts, backfill, schema drift and logging

Part A runs the pipeline on the small files in `test_data/` inside a separate
schema (`world_bank_test`), so the real tables are not touched. Every check prints
PASS/FAIL, and the last cell fails if any check failed.

Part B runs read-only checks on the real tables (`world_bank`) after the full and
incremental loads have been run with `run_pipeline`.

In [0]:
%run ./pipeline_utils

In [0]:
TEST_SCHEMA = "world_bank_test"
REAL_SCHEMA = DEFAULT_SCHEMA

test_tables = get_tables("", TEST_SCHEMA)
drop_all_tables(test_tables)      # start every test run from empty tables
create_tables(test_tables)

results = []

def check(name, condition, detail=""):
    results.append((name, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'} - {name} {detail}")

def step_log(run_id, layer):
    rows = (spark.table(test_tables["logs"])
            .filter((F.col("run_id") == run_id) & (F.col("layer") == layer))
            .collect())
    return rows[0] if rows else None

def silver_value(country_key, indicator, year):
    rows = (spark.table(test_tables["silver"])
            .filter((F.col("country_key") == country_key) & (F.col("indicator_code") == indicator) & (F.col("year") == year))
            .select("value").collect())
    return rows[0]["value"] if rows else None

def silver_count():
    return spark.table(test_tables["silver"]).count()

def duplicate_keys(table):
    return spark.table(table).groupBy(*SILVER_KEY).count().filter("count > 1").count()

### Test 1: initial (full) load
34 records: 3 countries x 2 indicators x 5 years, plus 4 records for Niue and
Tokelau, which have an empty country id in the API.

In [0]:
print('add_bronze_metadata' in dir())
print([n for n in dir() if 'metadata' in n.lower() or n.startswith('add_')])

In [0]:
run1, _ = run_pipeline(load_type="full", source_path="test_data/sample_baseline.json",
                       batch_id="test_full", schema=TEST_SCHEMA)

b, s = step_log(run1, "raw_to_bronze"), step_log(run1, "bronze_to_silver")
check("bronze inserted 34 rows", b["rows_inserted"] == 34, f"(got {b['rows_inserted']})")
check("silver inserted 34 rows", s["rows_inserted"] == 34, f"(got {s['rows_inserted']})")
check("silver has 34 rows", silver_count() == 34, f"(got {silver_count()})")
check("entities with empty country id are kept separately",
      spark.table(test_tables["silver"]).filter(F.col("country_key").isin("NAME_NIUE", "NAME_TOKELAU")).count() == 4)
check("every silver row has load_timestamp",
      spark.table(test_tables["silver"]).filter(F.col("load_timestamp").isNull()).count() == 0)

### Test 2: idempotency, re-running the exact same load

In [0]:
silver_before = spark.table(test_tables["silver"]).select(*SILVER_KEY, "value", "load_timestamp").collect()

run2, _ = run_pipeline(load_type="full", source_path="test_data/sample_baseline.json",
                       batch_id="test_full", schema=TEST_SCHEMA)

b, s = step_log(run2, "raw_to_bronze"), step_log(run2, "bronze_to_silver")
silver_after = spark.table(test_tables["silver"]).select(*SILVER_KEY, "value", "load_timestamp").collect()

check("re-run: bronze inserted 0", b["rows_inserted"] == 0, f"(got {b['rows_inserted']})")
check("re-run: silver inserted 0 and updated 0",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("re-run: silver table unchanged", sorted(silver_before) == sorted(silver_after))
check("no duplicate keys in silver", duplicate_keys(test_tables["silver"]) == 0)

### Test 3: incremental load with new and revised records (MERGE upsert)
`sample_revisions.json` has 7 records:
3 revised values, 1 value that was missing before (null -> value), 2 unchanged
records and 1 new year (2025).

In [0]:
run3, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_revisions.json",
                       batch_id="test_incremental_1", schema=TEST_SCHEMA)

s = step_log(run3, "bronze_to_silver")
check("incremental: 1 row inserted", s["rows_inserted"] == 1, f"(got {s['rows_inserted']})")
check("incremental: 4 rows updated", s["rows_updated"] == 4, f"(got {s['rows_updated']})")
check("silver has 35 rows", silver_count() == 35, f"(got {silver_count()})")
check("revised value applied (PK electricity 2023 = 95.8)", silver_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)
check("newly reported value applied (PK forest 2024 = 4.62)", silver_value("PK", "AG.LND.FRST.ZS", 2024) == 4.62)

rev = {r["change_type"]: r["count"] for r in
       spark.table(test_tables["revisions"]).groupBy("change_type").count().collect()}
check("revisions table: 3 value_revised + 1 newly_reported",
      rev.get("value_revised") == 3 and rev.get("newly_reported") == 1, f"(got {rev})")

display(spark.table(test_tables["revisions"]))

In [0]:
# Re-running the same incremental batch must not change anything
run3b, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_revisions.json",
                        batch_id="test_incremental_1", schema=TEST_SCHEMA)
s = step_log(run3b, "bronze_to_silver")
check("incremental re-run: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("incremental re-run: no extra revisions", spark.table(test_tables["revisions"]).count() == 4)

### Test 4: backfill, re-processing an older batch from Bronze to Silver
Re-running the old `test_full` batch must not overwrite the newer revised values.

In [0]:
run4, _ = run_pipeline(load_type="full", batch_id="test_full", layer="silver", schema=TEST_SCHEMA)
s = step_log(run4, "bronze_to_silver")
check("backfill: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")
check("backfill: newer value kept (95.8)", silver_value("PK", "EG.ELC.ACCS.ZS", 2023) == 95.8)

# Full rebuild of Silver from every Bronze batch also gives the same result
run4b, _ = run_pipeline(load_type="full", batch_id="ALL", layer="silver", schema=TEST_SCHEMA)
s = step_log(run4b, "bronze_to_silver")
check("rebuild from ALL batches: 0 inserted / 0 updated",
      s["rows_inserted"] == 0 and s["rows_updated"] == 0, f"(got {s['rows_inserted']}/{s['rows_updated']})")

### Test 5: schema drift and bad records
`sample_schema_drift.json` has 6 records:
1. a new, unexpected column (`footnote`) -> loaded, extra field kept in `_rescued_data`
2. `value` sent as text `"N/A"` -> quarantined in Bronze->Silver (value_not_numeric)
3. `date` = `"20X5"` -> quarantined in Bronze->Silver (invalid_year)
4. `indicator` object missing -> quarantined in Raw->Bronze (missing_indicator_id)
5. `country` sent as a plain string instead of an object -> quarantined in Raw->Bronze (missing_country)
6. a normal valid record -> loaded

The batch must finish without crashing.

In [0]:
run5, _ = run_pipeline(load_type="incremental", source_path="test_data/sample_schema_drift.json",
                       batch_id="test_drift_1", schema=TEST_SCHEMA)

b, s = step_log(run5, "raw_to_bronze"), step_log(run5, "bronze_to_silver")
check("drift batch finished with SUCCESS", b["status"] == "SUCCESS" and s["status"] == "SUCCESS")
check("raw->bronze: 6 read, 4 inserted, 2 quarantined",
      (b["rows_read"], b["rows_inserted"], b["rows_quarantined"]) == (6, 4, 2),
      f"(got {b['rows_read']}, {b['rows_inserted']}, {b['rows_quarantined']})")
check("bronze->silver: 4 read, 2 inserted, 2 quarantined",
      (s["rows_read"], s["rows_inserted"], s["rows_quarantined"]) == (4, 2, 2),
      f"(got {s['rows_read']}, {s['rows_inserted']}, {s['rows_quarantined']})")
check("new column captured in _rescued_data",
      spark.table(test_tables["bronze"]).filter(F.col("_rescued_data").contains("footnote")).count() == 1)
check("silver has 37 rows", silver_count() == 37, f"(got {silver_count()})")

display(spark.table(test_tables["quarantine"]).orderBy("layer", "reason"))
display(spark.table(test_tables["bronze"]).filter(F.col("_rescued_data").isNotNull())
        .select("indicator", "country", "date", "value", "_rescued_data", "batch_id"))

In [0]:
# Re-running the drift batch does not duplicate quarantine rows
quarantine_before = spark.table(test_tables["quarantine"]).count()
run_pipeline(load_type="incremental", source_path="test_data/sample_schema_drift.json",
             batch_id="test_drift_1", schema=TEST_SCHEMA)
check("quarantine not duplicated on re-run", spark.table(test_tables["quarantine"]).count() == quarantine_before)

### Test 6: a failing step is logged as FAILED

In [0]:
try:
    run_pipeline(load_type="incremental", source_path="test_data/file_that_does_not_exist.json",
                 batch_id="test_missing_file", schema=TEST_SCHEMA)
    raised = False
except RuntimeError as e:
    raised = True
    print("Expected error:", e)

failed_logs = (spark.table(test_tables["logs"])
               .filter((F.col("batch_id") == "test_missing_file") & (F.col("status") == "FAILED")).count())
check("missing file raises an error", raised)
check("missing file logged with status FAILED", failed_logs >= 1)

### Execution log of all test runs

In [0]:
display(spark.table(test_tables["logs"]).orderBy("start_time"))

## Part B: checks on the real tables
Run after loading the real data with `run_pipeline` (full load, then incremental).

In [0]:
real_tables = get_tables("", REAL_SCHEMA)

if spark.catalog.tableExists(real_tables["silver"]):
    check("real silver: no duplicate keys", duplicate_keys(real_tables["silver"]) == 0)
    check("real silver: no null load_timestamp",
          spark.table(real_tables["silver"]).filter(F.col("load_timestamp").isNull()).count() == 0)
    check("real bronze: no null load_timestamp",
          spark.table(real_tables["bronze"]).filter(F.col("load_timestamp").isNull()).count() == 0)

    display(spark.sql(f"""
        SELECT batch_id, load_type, source_file, COUNT(*) AS rows, MIN(load_timestamp) AS loaded_at
        FROM {real_tables['bronze']} GROUP BY batch_id, load_type, source_file ORDER BY loaded_at
    """))
    display(spark.sql(f"""
        SELECT COUNT(*) AS silver_rows,
               SUM(CASE WHEN is_reported THEN 1 ELSE 0 END) AS reported_values,
               COUNT(DISTINCT indicator_code) AS indicators,
               COUNT(DISTINCT country_key) AS countries,
               MIN(year) AS first_year, MAX(year) AS last_year
        FROM {real_tables['silver']}
    """))
    display(spark.table(real_tables["logs"]).orderBy(F.col("start_time").desc()))
else:
    print(f"{real_tables['silver']} does not exist yet, run run_pipeline first.")

In [0]:
failed = [name for name, ok in results if not ok]
print(f"{len(results) - len(failed)} / {len(results)} checks passed")
assert not failed, f"Failed checks: {failed}"